# Hazard zones of Nepal's districts

Zones: `very high`, `high`, `moderate`, `low`, `very low`.

Result: new column `hazard_zone` in `nepal_districts`.

## 1. Setup and database connection

In [9]:
import os
import re
from difflib import SequenceMatcher

import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()  # reads variables from .env (never committed / shared)

db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)

HAZARD_CSV = r"C:\Users\MarcComagic\OneDrive - Universität Zürich UZH\Masterarbeit\Data\district_hazard_zones\t0003-10.1080_19475705.2021.1879941.csv"

## 2. Load districts

Matching is performed using `name:en` (fallback `name`); `alt_name:en` and `alt_name` serve as additional name variants.

In [10]:
districts = pd.read_sql(
    """
    SELECT osm_id,
           name,
           COALESCE(tags -> 'name:en', name) AS name_en,
           tags -> 'alt_name:en'             AS alt_name_en,
           tags -> 'alt_name'                AS alt_name
    FROM nepal_districts
    """,
    engine,
)
print("Districts:", len(districts), "| unique osm_id:", districts["osm_id"].nunique())

Districts: 77 | unique osm_id: 77


## 3. Hazard zones from the source

The CSV contains a comma-separated list of districts for each zone. It is split into one row per district; the number of districts per zone is checked against the column "Number of districts".

In [11]:
zones_raw = pd.read_csv(HAZARD_CSV)

source = (
    zones_raw.assign(src_district=zones_raw["Districts"].str.split(","))
    .explode("src_district")
    .assign(
        src_district=lambda d: d["src_district"].str.strip(),
        hazard_zone=lambda d: d["Level of multi-hazard"].str.strip().str.lower(),
    )[["src_district", "hazard_zone"]]
    .reset_index(drop=True)
)

counts = source["hazard_zone"].value_counts()
expected = zones_raw.set_index(zones_raw["Level of multi-hazard"].str.lower())["Number of districts"]
assert (counts.reindex(expected.index) == expected).all(), "Number of districts per zone does not match the source"
assert not source["src_district"].duplicated().any(), "District listed more than once in the source"
print("Districts in the source:", len(source))
counts

Districts in the source: 75


hazard_zone
moderate     30
low          25
high         11
very high     7
very low      2
Name: count, dtype: int64

## 4. Normalise names

Transliterations vary (Kavrepalanchowk/Kavrepalanchok, Udaypur/Udayapur, Kanchannpur/Kanchanpur). As in `municipality_types.ipynb`, names are reduced to a "skeleton": lower case, bracketed additions and "District" removed, aspirated consonants simplified, v/w → b, sh → s, aa/ee/oo → a/i/u, double letters collapsed.

In [12]:
def skeleton(s):
    s = re.sub(r"\(.*?\)", " ", s.lower())  # bracketed additions such as "(Nepal)"
    s = re.sub(r"\bdistrict\b", " ", s)
    s = "".join(re.findall(r"[a-z]+", s))
    for a, b in [("aa", "a"), ("ee", "i"), ("oo", "u"), ("w", "b"), ("v", "b"), ("sh", "s"),
                 ("ts", "c"), ("z", "j"), ("f", "p"), ("y", "i")]:
        s = s.replace(a, b)
    for _ in range(2):  # chh -> ch -> c
        s = re.sub(r"([bcdgjkpt])h", r"\1", s)
    return re.sub(r"(.)\1+", r"\1", s)


def name_variants(row):
    names = [row["name_en"], row["alt_name_en"], row["alt_name"], row["name"]]
    variants = {v.strip() for n in names if isinstance(n, str) for v in n.split(";")}
    return sorted({skeleton(v) for v in variants} - {""})


districts["name_keys"] = districts.apply(name_variants, axis=1)
source["name_key"] = source["src_district"].apply(skeleton)

## 5. Matching

**Split districts:** The source uses the old division into 75 districts. Rukum and Nawalparasi were split in 2015; their successors (Eastern/Western Rukum, Nawalparasi W/Nawalpur) lie entirely within the former district. They inherit its zone – the source determined the zone for the former district as a whole, and no classification is available for the individual parts. These four are assigned explicitly, as fuzzy matching (Nawalparasi W → Nawalparasi, Nawalpur → Nawalparasi) would otherwise hit or miss by chance.

**Remaining districts:** most similar source name across all name variants (similarity ≥ 0.8). Each source entry may only be assigned to one district.

In [13]:
NAME_MIN = 0.8
SPLIT_DISTRICTS = {  # name:en in OSM -> district of the old division
    "Eastern Rukum": "Rukum",
    "Western Rukum District": "Rukum",
    "Nawalparasi W": "Nawalparasi",
    "Nawalpur District": "Nawalparasi",
}
assert set(SPLIT_DISTRICTS) <= set(districts["name_en"]), "Split district not found in nepal_districts"


def match(row):
    if row["name_en"] in SPLIT_DISTRICTS:
        parent = SPLIT_DISTRICTS[row["name_en"]]
        return parent, 1.0, "split (zone of former district)"
    scores = source["name_key"].apply(lambda k: max(SequenceMatcher(None, n, k).ratio() for n in row["name_keys"]))
    if scores.max() < NAME_MIN:
        return None, scores.max(), "no match"
    return source.loc[scores.idxmax(), "src_district"], scores.max(), "name"


districts[["src_district", "match_score", "match_info"]] = districts.apply(match, axis=1, result_type="expand")

# Source entry matched by several districts (except split ones) -> unclear, set all of them to unknown
normal = districts["match_info"] == "name"
dup = normal & districts["src_district"].where(normal).duplicated(keep=False)
districts.loc[dup, "src_district"] = None
districts.loc[dup, "match_info"] = "source entry assigned twice"

districts = districts.merge(source[["src_district", "hazard_zone"]], on="src_district", how="left")
districts["hazard_zone"] = districts["hazard_zone"].fillna("unknown")
districts["hazard_zone"].value_counts()

hazard_zone
moderate     32
low          25
high         11
very high     7
very low      2
Name: count, dtype: int64

## 6. Checks

In [14]:
print(districts["match_info"].value_counts().to_string(), "\n")

# All non-exact name matches, split districts and unknown
diag_cols = ["osm_id", "name_en", "src_district", "match_score", "match_info", "hazard_zone"]
districts.loc[
    (districts["match_score"] < 1) | (districts["match_info"] != "name") | (districts["hazard_zone"] == "unknown"),
    diag_cols,
].sort_values("match_score")

match_info
name                               73
split (zone of former district)     4 



,osm_id,name_en,src_district,match_score,match_info,hazard_zone
66,-4589462,Udayapur,Udaypur,0.933333,name,moderate
48,-4583248,Kavrepalanchok,Kavrepalanchowk,0.962963,name,high
11,-10493724,Western Rukum District,Rukum,1.000000,split (zone of former district),moderate
12,-10624408,Eastern Rukum,Rukum,1.000000,split (zone of former district),moderate
15,-4589560,Nawalparasi W,Nawalparasi,1.000000,split (zone of former district),moderate
14,-10489606,Nawalpur District,Nawalparasi,1.000000,split (zone of former district),moderate


In [15]:
# Source entries that were not assigned to any district
source.loc[~source["src_district"].isin(districts["src_district"]), ["src_district", "hazard_zone"]]

,src_district,hazard_zone


## 7. Update `nepal_districts`

The column `hazard_zone` is added to the table in place (or overwritten if it already exists). This keeps the geometry, `tags` as hstore and all other columns unchanged.

In [16]:
with engine.begin() as conn:
    districts[["osm_id", "hazard_zone"]].to_sql("tmp_district_hazard_zone", conn, if_exists="replace", index=False)
    conn.execute(text("ALTER TABLE nepal_districts ADD COLUMN IF NOT EXISTS hazard_zone text"))
    conn.execute(text("UPDATE nepal_districts SET hazard_zone = 'unknown'"))
    conn.execute(text(
        "UPDATE nepal_districts t SET hazard_zone = z.hazard_zone "
        "FROM tmp_district_hazard_zone z WHERE t.osm_id = z.osm_id"
    ))
    conn.execute(text("DROP TABLE tmp_district_hazard_zone"))

pd.read_sql(
    "SELECT hazard_zone, COUNT(*) AS n FROM nepal_districts GROUP BY 1 ORDER BY 2 DESC",
    engine,
)

,hazard_zone,n
0,moderate,32
1,low,25
2,high,11
3,very high,7
4,very low,2
